# Setup — Imports & LLM

In [ ]:
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader, UnstructuredMarkdownLoader, UnstructuredHTMLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from typing import Optional
import ast
import json

load_dotenv()
llm = ChatGroq(model='llama-3.3-70b-versatile')
print('LLM ready')

# Embeddings Class (used for all RAG tasks)

In [ ]:
class Embeddings():
    def __init__(self):
        self._ef = DefaultEmbeddingFunction()
    def embed_documents(self, texts):
        return [[float(x) for x in v] for v in self._ef(texts)]
    def embed_query(self, text):
        return [float(x) for x in self._ef([text])[0]]

embeddings = Embeddings()
print('Embeddings ready')

# Task 2a — RAG with Wikipedia (Web)

In [ ]:
# Load
web_docs = WebBaseLoader('https://en.wikipedia.org/wiki/One_Piece').load()
print(f'Loaded {len(web_docs)} web documents')

# Chunk
web_chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(web_docs)
print(f'Created {len(web_chunks)} chunks')

# Store
web_store = Chroma.from_documents(web_chunks, embeddings, collection_name='web_rag')
web_retriever = web_store.as_retriever(search_kwargs={'k': 3})
print('Web vector store ready')

In [ ]:
# Prompt + Chain
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Answer using only this context: {context}'),
    ('human', '{question}')
])
chain = prompt | llm | StrOutputParser()

def ask(question, retriever):
    docs = retriever.invoke(question)
    context = '\n\n'.join(d.page_content for d in docs)
    return chain.invoke({'context': context, 'question': question})

print(ask('who is Luffy?', web_retriever))

# Task 2b — RAG with Markdown File

In [ ]:
# Load
md_docs = UnstructuredMarkdownLoader('new.md').load()
print(f'Loaded {len(md_docs)} markdown documents')

# Chunk
md_chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(md_docs)
print(f'Created {len(md_chunks)} chunks')

# Store
md_store = Chroma.from_documents(md_chunks, embeddings, collection_name='markdown_rag')
md_retriever = md_store.as_retriever(search_kwargs={'k': 3})
print('Markdown vector store ready')

print(ask('give me a summary of One Piece', md_retriever))

# Task 2c — RAG with HTML File

In [ ]:
# First create a sample HTML file
html_content = '''
<!DOCTYPE html>
<html>
<body>
<h1>One Piece</h1>
<p>One Piece is a Japanese manga series written and illustrated by Eiichiro Oda.</p>
<p>The story follows Monkey D. Luffy, a boy whose body gained the properties of rubber after eating a Devil Fruit.</p>
<p>Luffy explores the ocean in search of the world's ultimate treasure known as the One Piece.</p>
<p>He wants to become the next King of the Pirates.</p>
<h2>Main Characters</h2>
<p>Luffy - Captain of the Straw Hat Pirates</p>
<p>Zoro - Swordsman of the crew</p>
<p>Nami - Navigator of the crew</p>
</body>
</html>
'''

with open('sample.html', 'w') as f:
    f.write(html_content)
print('HTML file created')

# Load
html_docs = UnstructuredHTMLLoader('sample.html').load()
print(f'Loaded {len(html_docs)} html documents')

# Chunk
html_chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(html_docs)
print(f'Created {len(html_chunks)} chunks')

# Store
html_store = Chroma.from_documents(html_chunks, embeddings, collection_name='html_rag')
html_retriever = html_store.as_retriever(search_kwargs={'k': 3})
print('HTML vector store ready')

print(ask('who is Zoro?', html_retriever))

# Task 1 — Prompt Engineering (5 Bad → 5 Good)

In [ ]:
bad_prompt_1 = 'explain me the concept of quantum computing'
good_prompt_1 = '''
I am a beginner who knows the basics of physics.
Explain quantum computing starting from the basics and gradually increase the difficulty.
Present it as a story that anyone can understand, with real-world analogies.
'''

bad_prompt_2 = 'explain me the rules of football'
good_prompt_2 = '''
I watch football daily but I don't fully understand all the rules.
Explain the offside rule and penalty rules clearly.
Format it as a numbered list with examples for each rule.
'''

bad_prompt_3 = 'explain genai'
good_prompt_3 = '''
I am a software developer with basic Python knowledge but no AI experience.
Explain Generative AI from scratch — what it is, how it works, and real world use cases.
Format it as short paragraphs with a heading for each section.
'''

bad_prompt_4 = 'what is the nearest planet to earth'
good_prompt_4 = '''
I am a curious student who just started learning astronomy.
Tell me which planet is nearest to Earth on average, why it changes,
and how scientists calculate this.
Explain it simply with an analogy, not just facts.
'''

bad_prompt_5 = 'how to be a professional footballer'
good_prompt_5 = '''
I am a 19 year old who plays football casually on weekends.
Give me a realistic step by step roadmap to become a professional footballer —
covering fitness, skill training, trials, and mindset.
Format it as a weekly action plan.
'''

prompts = [good_prompt_1, good_prompt_2, good_prompt_3, good_prompt_4, good_prompt_5]

for i, p in enumerate(prompts, 1):
    print(f'\n--- Prompt {i} ---')
    print(llm.invoke(p).content)

# Task 3 — Chain of Thought Math Problem

In [ ]:
bad_math_prompt = 'solve 2x + 3y = 300'

good_math_prompt = '''
Solve the equation: 2x + 3y = 300

Think step by step:
1. First explain what is given in the problem
2. Identify the variables and constants separately
3. Show every calculation clearly
4. Explain your reasoning at each step before moving to the next
5. Give the final answer with explanation
'''

print(llm.invoke(good_math_prompt).content)

# Task 4 — Self-Reflecting Code Review Agent using AST

In [ ]:
bad_code = '''
def calculate_average(numbers):
    total = 0
    for num in numbers:
        total += num
    return total / len(numbers)
'''

# Step 1: AST parses the code
try:
    tree = ast.parse(bad_code)
    ast_result = 'No syntax errors found'
except SyntaxError as e:
    ast_result = f'Syntax error found: {e}'

print(f'AST Result: {ast_result}')

# Step 2: LLM reflects on the code
review_prompt = f'''
You are a self-reflecting code review agent.

Code to review:
{bad_code}

AST Parser Result: {ast_result}

Step 1: Explain what is wrong
Step 2: Reflect — are there any deeper issues beyond what AST found?
Step 3: Write the corrected code
'''

print(llm.invoke(review_prompt).content)